In [104]:
# Import libraries
import pandas as pd 
import numpy as np 
import math 
import statistics 
import matplotlib.pyplot as plt 

**Clean First Dataset**:

In [111]:
# Import Subset data and create dataframe
raw_df = pd.read_csv(r".\Signal_data\Subset_2025-2026.csv", sep=";", engine="python") 

# Drop non-ES columns
raw_df_1 = raw_df[["Date (UTC)", "ES_DIR", "ES_Open"]].copy()

# Rename for convention
raw_df_1.columns = ["Date", "Signal", "Entry_Price"]

# Set Index
raw_df_1["Date"] = pd.to_datetime(raw_df_1["Date"]) + pd.Timedelta(days=1)
raw_df_1 = raw_df_1[~raw_df_1.index.duplicated(keep="last")]
raw_df_1 = raw_df_1.set_index("Date")
raw_df_1 = raw_df_1[raw_df_1.index.notna()]
raw_df_1 = raw_df_1.sort_index() 

# Change variable types
raw_df_1["Signal"] = raw_df_1["Signal"].fillna(0).astype("int64")
raw_df_1["Entry_Price"] = (raw_df_1["Entry_Price"].astype(str).str.replace(",", ".", regex=False))
raw_df_1["Entry_Price"] = pd.to_numeric(raw_df_1["Entry_Price"], errors="coerce")

# Preview
raw_df_1

,Signal,Entry_Price
Date,,
2025-06-23,1,5964.00
2025-06-24,1,6078.00
2025-06-25,1,6144.75
2025-06-26,1,6144.75
2025-06-27,1,6197.50
...,...,...
2026-03-16,1,6608.50
2026-03-17,1,6700.00
2026-03-18,1,6722.50


**Clean Second Dataset**:

In [110]:
# Import Q2_2026 data
raw_df = pd.read_csv(r".\Signal_data\Q2_2026-06-02.csv", engine="python") 

# Drop non-ES columns
raw_df_2 = raw_df[["utc_date", "ES_classification", "ES_refOpen"]].copy()

# Rename for convention
raw_df_2.columns = ["Date", "Signal", "Entry_Price"]

# Handle index
raw_df_2["Date"] = pd.to_datetime(raw_df_2["Date"], errors="coerce", dayfirst=True)
raw_df_2 = raw_df_2.dropna(subset=["Date"])
raw_df_2 = raw_df_2.set_index("Date") 
raw_df_2 = raw_df_2.sort_index() 

# Change string signal to polar signal 
mapping = {np.nan:0, "BULLISH":1, "BEARISH":-1} 
raw_df_2["Clean_Signal"] = raw_df_2["Signal"].map(mapping)
raw_df_2["Clean_Signal"] = raw_df_2["Clean_Signal"].ffill().fillna(0).astype(int) 
raw_df_2.drop(columns=["Signal"], inplace = True)
raw_df_2.rename(columns={"Clean_Signal":"Signal"}, inplace=True)
raw_df_2 = raw_df_2[["Signal", "Entry_Price"]]

# Preview
raw_df_2

,Signal,Entry_Price
Date,,
2026-03-29,0,6380.00
2026-03-30,1,6385.25
2026-03-31,1,6562.00
2026-04-01,1,6618.50
2026-04-02,-1,6622.50
2026-04-05,1,6590.00
2026-04-06,1,6650.50
2026-04-07,1,6689.00
2026-04-08,1,6814.25


**Merge Datasets**:

In [112]:
# Merge to dataframes of trade history
raw_df_3 = pd.concat([raw_df_1, raw_df_2], axis=0, sort=False)

# Save to csv
raw_df_3.reset_index().to_csv(r".\Signal_data\ES_2025-2026.csv", index=False)

#Preview
raw_df_3

,Signal,Entry_Price
Date,,
2025-06-23,1,5964.00
2025-06-24,1,6078.00
2025-06-25,1,6144.75
2025-06-26,1,6144.75
2025-06-27,1,6197.50
...,...,...
2026-06-09,0,NaN
2026-06-10,0,NaN
2026-06-11,0,NaN
